# VAYU Climate Digital Twin — Kaggle GPU Training (Western Ghats) v2

**Accelerator**: GPU T4 x2 (recommended) or T4 x1 / P100  
**Target**: R2_tmax >= 0.80, R2_rain >= 0.20 — Western Ghats, leakage-safe calendar split

## What changed in v2
- **Real static features**: DEM (Copernicus 90m) and land/sea mask (ESA WorldCover 2021)
  replace the previous synthetic elevation ridge / geometric coastline. Already baked
  into the sequence tensors — **no preprocessing runs on Kaggle anymore**, sequences
  are pre-built and just copied into place.
- **Leakage-safe calendar split**: train 2010–2021, validation 2022, held-out test
  2023–2025 (previously an 85/15 positional split with no test set).
- **Train-only normalization**: z-score statistics are fit on 2010–2021 only.
- **Ghats ridge smoothness exemption**: now derived from real land mask + lat/lon
  (399/1311 nodes), not a raw longitude stripe that could bleed into other regions.
- **Held-out test evaluation runs automatically** after training and writes
  `test_report.json` with R²/RMSE/MAE + skill vs persistence/climatology.

## Prior best result (v1, synthetic static, no held-out test)
R2_tmax=0.817, R2_tmin=0.809, R2_rain=0.200 (epoch 27/60, `vayu_best (2).pt`).
This is the number to beat with real terrain + leakage-safe evaluation.

## Region priority: rainfall (extreme orographic rainfall)
Western Ghats is the flagship extreme-orographic-rainfall region (2018/2019/2024
Kerala floods). Rain weight raised 1.8 -> 2.2 per arXiv:2509.23267, arXiv:2605.30122,
arXiv:2402.01295.

## Required Dataset (Add Input -> Search by name)
Upload a **new version** of `shyam31415/vayu-western-ghats-processed-v1` containing:
- `train_sequences.pt`, `val_sequences.pt`, `test_sequences.pt` (17 features/node)
- `norm_params_2010-2025.nc`, `sequence_manifest.json`

## Steps
1. Enable GPU: Settings -> Accelerator -> **GPU T4 x2**
2. Add the v2 dataset above via "Add Input"
3. Run all cells top to bottom


In [ ]:
# ── Environment check ──────────────────────────────────────────────────
import subprocess, sys, os

result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout or 'No GPU detected — switch accelerator to GPU in Settings!')
print('Python:', sys.version)

In [ ]:
# ── Install dependencies ──────────────────────────────────────────────
# torch-geometric version must match torch; use 2.5.3 for torch 2.x on Kaggle.
!pip install -q torch-geometric==2.5.3 xarray netcdf4 typer scipy
print('Dependencies installed')

In [ ]:
# ── Mount project code and locate dataset ─────────────────────────────────
import sys, os
from pathlib import Path

REGION = 'western_ghats'
REPO_DIR = '/kaggle/working/isro'
PROCESSED_DIR = f'{REPO_DIR}/data/processed_{REGION}'
CHECKPOINT_DIR = f'{REPO_DIR}/checkpoints/wg_main'

# Clone or pull — check for .git to detect a broken/partial directory
if os.path.exists(f'{REPO_DIR}/.git'):
    os.system(f'git -C {REPO_DIR} pull')
else:
    os.system(f'rm -rf {REPO_DIR}')   # remove partial/broken dir if any
    os.system(f'git clone https://github.com/Shyamistic/vayu.git {REPO_DIR}')

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print('Working dir:', os.getcwd())

# NOTE: must run after clone/rm -rf above, since PROCESSED_DIR/CHECKPOINT_DIR
# are nested inside REPO_DIR and would otherwise be wiped out by rm -rf.
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

# Locate the uploaded Kaggle dataset (vayu-western-ghats-processed-v1, v2)
root = Path('/kaggle/input')
required = ['train_sequences.pt', 'val_sequences.pt', 'test_sequences.pt', 'sequence_manifest.json']
found = {name: next(iter(root.rglob(name)), None) for name in required}
missing = [k for k, v in found.items() if v is None]
if missing:
    raise RuntimeError("Missing dataset files: " + ", ".join(missing) +
                       ". Attach the v2 dataset via 'Add Input'.")
parent_counts = {}
for p in found.values():
    parent_counts[str(p.parent)] = parent_counts.get(str(p.parent), 0) + 1
DATASET_DIR = max(parent_counts, key=parent_counts.get)
print('Dataset dir:', DATASET_DIR)


In [ ]:
# ── Copy pre-built sequences into place (no preprocessing needed — already built) ──
import shutil
from pathlib import Path as _P

for f in ['train_sequences.pt', 'val_sequences.pt', 'test_sequences.pt',
          'norm_params_2010-2025.nc', 'sequence_manifest.json']:
    src = _P(DATASET_DIR) / f
    if src.exists():
        shutil.copy(src, PROCESSED_DIR)
        print(f'copied {f}')
    else:
        print(f'skip (not in bundle): {f}')

os.system(f'ls -lah {PROCESSED_DIR}')


In [ ]:
# -- Smoke check: verify model and data before full training ----------------
import subprocess, sys, torch, json
PY = sys.executable

manifest = json.loads((_P(PROCESSED_DIR) / 'sequence_manifest.json').read_text())
print('Splits:', {k: v['saved_sequences'] for k, v in manifest['splits'].items()})
print('Feature count:', manifest['feature_count'])

# ── Pre-flight: verify sequences have 17 features (catches stale files) ──
_seq_path = f'{PROCESSED_DIR}/train_sequences.pt'
_seqs = torch.load(_seq_path, map_location='cpu', weights_only=False)
_nf = _seqs[0][0].x.shape[-1]  # (num_nodes, seq_len, features)
print(f'Sequence feature count: {_nf} (expected 17)')
if _nf != 17:
    raise RuntimeError(
        f'STALE SEQUENCES: found {_nf} features, expected 17. '
        f'Re-upload the v2 dataset with 17-feature sequences.'
    )
del _seqs  # free memory
print('✓ Sequences verified: 17 features per node')

_r = subprocess.run([PY,'-m','ai_engine.trainer',
    '--data-dir', PROCESSED_DIR,
    '--checkpoint-dir', f'{REPO_DIR}/checkpoints/wg_smoke',
    '--epochs','1','--device','auto','--smoke-only'],
    cwd=REPO_DIR, capture_output=True, text=True)
print(_r.stdout[-3000:] if _r.stdout else '')
if _r.returncode != 0:
    print('\n=== smoke STDERR ===')
    print(_r.stderr[-4000:] if _r.stderr else '(empty)')
    raise RuntimeError(f'Smoke check failed (exit {_r.returncode})')
print('\n✓ Smoke check PASSED — model + data + loss all wired correctly')


In [ ]:
# ── FINAL best-quality training run ──────────────────────────────────
# Target: R²_tmax >= 0.80, R²_rain >= 0.20 on the held-out 2023-2025 test set.
# --rain-weight 2.2 biases the loss toward this region's orographic rainfall
# priority. Held-out test evaluation runs automatically at the end of training
# since test_sequences.pt is present, writing test_report.json.
import os, subprocess, sys
PY = sys.executable
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

subprocess.run([PY,'-m','ai_engine.trainer',
    '--data-dir',               PROCESSED_DIR,
    '--checkpoint-dir',         CHECKPOINT_DIR,
    '--epochs',                 '100',
    '--device',                 'auto',
    '--amp',
    '--batch-size',             '1',
    '--grad-accum-steps',       '8',
    '--cosine-lr',
    '--early-stopping-patience','20',
    '--weight-decay',           '1e-4',
    '--gnn-dropout',            '0.12',
    '--lambda-conservation',    '0.02',
    '--lambda-smoothness',      '0.02',
    '--rain-weight',            '2.2',
    '--norm-params-file',       f'{PROCESSED_DIR}/norm_params_2010-2025.nc',
    '--run-baselines',
    '--require-benchmarks'],
    check=True, cwd=REPO_DIR)

os.system(f'ls -lah {CHECKPOINT_DIR}')


In [ ]:
# ── Load history and plot training curves ────────────────────────────────
import json, matplotlib.pyplot as plt
from pathlib import Path

history_path = Path(CHECKPOINT_DIR) / 'training_history.json'
if not history_path.exists():
    print('No training_history.json yet — run the training cell first.')
else:
    history = json.loads(history_path.read_text())

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(history['epochs'], history['train_loss'], label='Train Loss')
    axes[0].plot(history['epochs'], history['val_loss'],   label='Val Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].set_title('VAYU Training Loss')
    axes[0].legend()
    axes[0].grid(True)

    axes[1].plot(history['epochs'], history['val_r2'], color='green', label='R² Tmax')
    axes[1].axhline(0.80, color='red', linestyle='--', label='Target R²=0.80')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('R²')
    axes[1].set_title('Validation R²')
    axes[1].legend()
    axes[1].grid(True)

    plt.tight_layout()
    plt.savefig('/kaggle/working/training_curves.png', dpi=150)
    plt.show()
    print('Best val_loss:', min(history['val_loss']))
    if history['benchmark_metrics']:
        last = history['benchmark_metrics'][-1]
        print(f"Latest validation R2_tmax={last.get('r2_tmax'):.3f} | R2_tmin={last.get('r2_tmin'):.3f} | R2_rain={last.get('r2_rain'):.3f}")


In [ ]:
# ── Held-out test set results (2023–2025, never seen during training/validation) ──
import json
from pathlib import Path

test_report_path = Path(CHECKPOINT_DIR) / 'test_report.json'
if test_report_path.exists():
    test_results = json.loads(test_report_path.read_text())
    for var, metrics in test_results.items():
        print(f"{var}: R2={metrics['r2']:.3f} | RMSE={metrics['rmse']:.3f} | MAE={metrics['mae']:.3f} | "
              f"skill_vs_persistence={metrics['skill_vs_persistence']:.3f} | skill_vs_climatology={metrics['skill_vs_climatology']:.3f}")
    print()
    print('Prior v1 result (synthetic static, no held-out test): R2_tmax=0.817, R2_tmin=0.809, R2_rain=0.200')
else:
    print('No test_report.json found — ensure test_sequences.pt was present before training.')

In [ ]:
# ── Save best checkpoint for download ──────────────────────────────────
import shutil
from pathlib import Path

best_ckpt = Path(CHECKPOINT_DIR) / 'vayu_best.pt'
if best_ckpt.exists():
    shutil.copy(best_ckpt, '/kaggle/working/vayu_best.pt')
    size_mb = best_ckpt.stat().st_size / 1e6
    print(f'Checkpoint ready: /kaggle/working/vayu_best.pt ({size_mb:.1f} MB)')
    print('Download and upload to S3:')
    print('  aws s3 cp vayu_best.pt s3://vayu-models/checkpoints/')
else:
    print('vayu_best.pt not found — check training cell output for errors.')

## Next Steps After Training

1. **Download** `vayu_best.pt` from Kaggle Output
2. **Upload to S3**:
   ```bash
   aws s3 cp vayu_best.pt s3://vayu-climate-models/checkpoints/vayu_best.pt
   ```
3. **Trigger ECS deployment** (CDK will mount S3 checkpoint automatically)
4. **Verify**: `curl https://api.vayu-climate.com/health`

## Kaggle Quota Tips
- Each run ~ 2-4 hours on T4 (30h/week quota)
- Use `early_stopping_patience=20` to auto-stop when converged
- Enable Accelerator **T4 x2** for 2x speed if available